In [1]:
from pathlib import Path

ruta_raw = Path("../data/raw")
print(ruta_raw.resolve())
print(list(ruta_raw.glob("*.csv")))

C:\Users\Juan David Salazar T\Desktop\Estudios\proyecto-ciencia-datos\data\raw
[WindowsPath('../data/raw/incidentes_viales.csv'), WindowsPath('../data/raw/Mede_Victimas_inci.csv')]


Leer los archivos

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

ruta_raw = Path("../data/raw")

incidentes_raw = pd.read_csv(ruta_raw / "incidentes_viales.csv", low_memory=False)

partes = []
for parte in pd.read_csv(ruta_raw / "Mede_Victimas_inci.csv", chunksize=100_000, dtype=str):
    partes.append(parte.dropna(how="all"))
victimas_raw = pd.concat(partes, ignore_index=True)

print("incidentes:", incidentes_raw.shape)
print("victimas:  ", victimas_raw.shape)

incidentes: (270765, 18)
victimas:   (235843, 19)


Duplicados exactos: detectar con duplicated y eliminar con drop_duplicates

In [12]:
duplicados = incidentes_raw[incidentes_raw.duplicated(keep=False)].sort_values('NRO_RADICADO')
print(len(duplicados))
duplicados[['NRO_RADICADO', 'FECHA_ACCIDENTE', 'CLASE_ACCIDENTE']].head(6)

13


,NRO_RADICADO,FECHA_ACCIDENTE,CLASE_ACCIDENTE
107206,48996,01/10/2017 05:10:00,Choque
233477,48996,01/10/2017 05:10:00,Choque
106514,51217,14/10/2017 05:47:00,Choque
137858,51217,14/10/2017 05:47:00,Choque
255587,58560,20/11/2016 02:05:00,Atropello
266852,58560,20/11/2016 02:05:00,Atropello


In [13]:
incidentes = incidentes_raw.drop_duplicates().reset_index(drop=True)
print(incidentes.shape, incidentes['NRO_RADICADO'].is_unique)

(270758, 18) False


Tipos de dato con astype

In [14]:
print('Antes:', round(incidentes.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

incidentes['AÑO'] = incidentes['AÑO'].astype(str).str.replace(r"\r", "", regex=False)

incidentes = incidentes.astype({
    'AÑO': 'int16', 'MES': 'int8',
    'X': 'float32', 'Y': 'float32',
})
victimas = victimas.astype({
    'Año': 'int16', 'Num_dia': 'int8',
})

print('Después:', round(incidentes.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 96.5 MB
Después: 89.6 MB


Nulos: comprobar la causa antes de decidir

In [15]:
nulos = incidentes['DISEÑO'].isna()
muertos = incidentes['GRAVEDAD_ACCIDENTE'] == 'Con muertos'
print((nulos == muertos).all())
print(muertos[nulos].all())

False
True


In [16]:
print(incidentes[incidentes['DISEÑO'].isna()][['GRAVEDAD_ACCIDENTE', 'AÑO']].value_counts())
incidentes['DISEÑO'] = incidentes['DISEÑO'].fillna('Sin información')
print(incidentes['DISEÑO'].isna().sum())

GRAVEDAD_ACCIDENTE  AÑO 
Con muertos         2015    265
                    2016    258
                    2017    245
                    2018    233
                    2014    140
Name: count, dtype: int64
0


Texto con los métodos .str

In [17]:
nombres = incidentes['BARRIO']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

incidentes['BARRIO'] = (
    incidentes['BARRIO']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
print(incidentes['BARRIO'].str.contains('  ', regex=False).sum(),
      incidentes['BARRIO'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 220
Espacio invisible: 0
0 0


Duplicados implícitos

In [18]:
conocidos = incidentes[incidentes['NRO_RADICADO'].notna()].copy()
conocidos['NRO_RADICADO'] = conocidos['NRO_RADICADO'].astype(str)
repetidos = conocidos[conocidos['NRO_RADICADO'].duplicated(keep=False)]
print(conocidos['NRO_RADICADO'].duplicated().sum())
print(repetidos['NRO_RADICADO'].isin(victimas['Radicado']).sum(), 'de', len(repetidos))

23
26 de 46


Limpieza reutilizable con funciones

In [19]:
PATRON_CAIDA = r'Ca[ií]da (de )?Ocupante'

def limpiar_texto(serie):
    return (serie.str.replace('\xa0', ' ', regex=False)
                 .str.replace(r'\s+', ' ', regex=True)
                 .str.strip())

def corregir_codificacion(serie):
    return serie.str.replace(r'\\x([0-9A-Fa-f]{2})', lambda m: chr(int(m.group(1), 16)), regex=True)

def limpiar_incidentes(df):
    df = df.drop_duplicates().reset_index(drop=True)

    df['AÑO'] = df['AÑO'].astype(str).str.replace(r'\r', '', regex=False)
    df = df.astype({'AÑO': 'int16', 'MES': 'int8', 'X': 'float32', 'Y': 'float32'})
    df['NRO_RADICADO'] = df['NRO_RADICADO'].astype(str).replace('nan', np.nan)

    duplicado_real = df.duplicated(subset=['NRO_RADICADO', 'EXPEDIENTE']) & df['EXPEDIENTE'].notna()
    df = df[~duplicado_real].reset_index(drop=True)

    for col in ['GRAVEDAD_ACCIDENTE', 'DISEÑO', 'BARRIO', 'COMUNA']:
        df[col] = corregir_codificacion(df[col])
    for col in ['BARRIO', 'DIRECCION']:
        df[col] = limpiar_texto(df[col])
    df['CLASE_ACCIDENTE'] = df['CLASE_ACCIDENTE'].str.replace(PATRON_CAIDA, 'Caída de ocupante', regex=True)
    df['NUMCOMUNA'] = df['NUMCOMUNA'].str.zfill(2)

    df['DISEÑO'] = df['DISEÑO'].fillna('Sin información')
    df['CLASE_ACCIDENTE'] = df['CLASE_ACCIDENTE'].fillna('Sin información')

    assert df.duplicated().sum() == 0
    assert df.dropna(subset=['EXPEDIENTE']).duplicated(subset=['NRO_RADICADO', 'EXPEDIENTE']).sum() == 0
    return df

def limpiar_victimas(df):
    df = df.drop_duplicates().reset_index(drop=True)

    for col in ['Año', 'Num_dia', 'Hora', 'Edad']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in ['Latitud', 'Longitud']:
        df[col] = pd.to_numeric(df[col].str.replace(',', '.', regex=False), errors='coerce')
    df = df.astype({'Año': 'int16', 'Num_dia': 'int8', 'Hora': 'Int8', 'Edad': 'Int16',
                    'Latitud': 'float32', 'Longitud': 'float32'})

    df['Barrio'] = limpiar_texto(df['Barrio'])
    df['Clase_incidente'] = df['Clase_incidente'].str.replace(PATRON_CAIDA, 'Caída de ocupante', regex=True)
    df['Sexo'] = df['Sexo'].str.title()
    df['Condicion'] = df['Condicion'].str.capitalize()
    df['Mes'] = df['Mes'].str[:3]

    partes = df['Comuna'].str.split(' - ', n=1, expand=True)
    df['NUMCOMUNA'] = partes[0]
    df['Comuna'] = partes[1]

    assert df.duplicated().sum() == 0
    return df

incidentes = limpiar_incidentes(incidentes_raw)
victimas = limpiar_victimas(victimas_raw)
print(incidentes.shape)
print(victimas.shape)

(270742, 18)
(235646, 20)


In [20]:
assert incidentes.shape == (270742, 18)
assert incidentes['COMUNA'].isna().sum() == 12796       # 2 nulos estaban entre los duplicados
assert incidentes['DISEÑO'].isna().sum() == 0
assert str(victimas['Edad'].dtype) == 'Int16'

Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Paso 1: recargar los datos crudos

In [21]:
RUTA = Path("../data/raw")

incidentes_crudo = pd.read_csv(RUTA / 'incidentes_viales.csv', low_memory=False)

partes = []
for parte in pd.read_csv(RUTA / 'Mede_Victimas_inci.csv', chunksize=100_000, dtype=str):
    partes.append(parte.dropna(how='all'))
victimas_crudo = pd.concat(partes, ignore_index=True)

print(incidentes_crudo.shape, victimas_crudo.shape)

(270765, 18) (235843, 19)


Paso 2: una función que anota el estado de la tabla

In [22]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

Paso 3: un paso de limpieza por función

In [23]:
PATRON_CAIDA = r'Ca[ií]da (de )?Ocupante'

def limpiar_texto(serie):
    return (serie.str.replace('\xa0', ' ', regex=False)
                 .str.replace(r'\s+', ' ', regex=True)
                 .str.strip())

def corregir_codificacion(serie):
    return serie.str.replace(r'\\x([0-9A-Fa-f]{2})', lambda m: chr(int(m.group(1), 16)), regex=True)

# ---------- pasos que sirven para las dos tablas ----------

def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

# ---------- pasos de incidentes ----------

def tipos_incidentes(df):
    df = df.copy()
    df['AÑO'] = df['AÑO'].astype(str).str.replace(r'\r', '', regex=False)
    df['NRO_RADICADO'] = df['NRO_RADICADO'].astype(str).replace('nan', np.nan)
    return df.astype({'AÑO': 'int16', 'MES': 'int8', 'X': 'float32', 'Y': 'float32'})

def quitar_accidentes_repetidos(df):
    repetido = df.duplicated(subset=['NRO_RADICADO', 'EXPEDIENTE']) & df['EXPEDIENTE'].notna()
    return df[~repetido].reset_index(drop=True)

def limpiar_texto_incidentes(df):
    df = df.copy()
    for col in ['GRAVEDAD_ACCIDENTE', 'DISEÑO', 'BARRIO', 'COMUNA']:
        df[col] = corregir_codificacion(df[col])
    for col in ['BARRIO', 'DIRECCION']:
        df[col] = limpiar_texto(df[col])
    df['CLASE_ACCIDENTE'] = df['CLASE_ACCIDENTE'].str.replace(PATRON_CAIDA, 'Caída de ocupante', regex=True)
    df['NUMCOMUNA'] = df['NUMCOMUNA'].str.zfill(2)
    return df

def rellenar_nulos_incidentes(df):
    df = df.copy()
    df['DISEÑO'] = df['DISEÑO'].fillna('Sin información')
    df['CLASE_ACCIDENTE'] = df['CLASE_ACCIDENTE'].fillna('Sin información')
    return df

# ---------- pasos de víctimas ----------

def tipos_victimas(df):
    df = df.copy()
    for col in ['Año', 'Num_dia', 'Hora', 'Edad']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in ['Latitud', 'Longitud']:
        df[col] = pd.to_numeric(df[col].str.replace(',', '.', regex=False), errors='coerce')
    return df.astype({'Año': 'int16', 'Num_dia': 'int8', 'Hora': 'Int8', 'Edad': 'Int16',
                      'Latitud': 'float32', 'Longitud': 'float32'})

def limpiar_texto_victimas(df):
    df = df.copy()
    df['Barrio'] = limpiar_texto(df['Barrio'])
    df['Clase_incidente'] = df['Clase_incidente'].str.replace(PATRON_CAIDA, 'Caída de ocupante', regex=True)
    df['Sexo'] = df['Sexo'].str.title()
    df['Condicion'] = df['Condicion'].str.capitalize()
    df['Mes'] = df['Mes'].str[:3]
    partes = df['Comuna'].str.split(' - ', n=1, expand=True)
    df['NUMCOMUNA'] = partes[0]
    df['Comuna'] = partes[1]
    return df

Paso 4: encadenar los pasos con pipe

In [24]:
registro = []

incidentes_limpio = (incidentes_crudo
    .pipe(anotar, 'incidentes', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'incidentes', '1 sin duplicados')
    .pipe(tipos_incidentes)
    .pipe(anotar, 'incidentes', '2 tipos')
    .pipe(quitar_accidentes_repetidos)
    .pipe(anotar, 'incidentes', '3 sin accidentes repetidos')
    .pipe(limpiar_texto_incidentes)
    .pipe(anotar, 'incidentes', '4 texto')
    .pipe(rellenar_nulos_incidentes)
    .pipe(anotar, 'incidentes', '5 nulos rellenados'))

victimas_limpio = (victimas_crudo
    .pipe(anotar, 'victimas', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'victimas', '1 sin duplicados')
    .pipe(tipos_victimas)
    .pipe(anotar, 'victimas', '2 tipos')
    .pipe(limpiar_texto_victimas)
    .pipe(anotar, 'victimas', '3 texto'))

print(incidentes_limpio.shape, victimas_limpio.shape)

(270742, 18) (235646, 20)


Paso 5: el informe de cambios

In [25]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,incidentes,0 crudo,270765,51620,96.3
1,incidentes,1 sin duplicados,270758,51607,96.5
2,incidentes,2 tipos,270758,51607,84.0
3,incidentes,3 sin accidentes repetidos,270742,51602,84.0
4,incidentes,4 texto,270742,51602,83.9
5,incidentes,5 nulos rellenados,270742,50457,83.9
6,victimas,0 crudo,235843,1143,68.4
7,victimas,1 sin duplicados,235646,1128,68.8
8,victimas,2 tipos,235646,44710,53.7
9,victimas,3 texto,235646,66422,55.1


Paso 6: validar el resultado

In [26]:
def validar(i, v):
    assert not i.dropna(subset=['EXPEDIENTE']).duplicated(subset=['NRO_RADICADO', 'EXPEDIENTE']).any(), 'accidente repetido'
    assert i['MES'].between(1, 12).all(), 'mes fuera de rango'
    assert i['AÑO'].between(2014, 2020).all(), 'año fuera de rango'
    assert i['DISEÑO'].notna().all(), 'accidentes sin diseño'
    assert not i['GRAVEDAD_ACCIDENTE'].str.contains(r'\\x', regex=True).any(), 'errores de codificación'
    assert not i['BARRIO'].str.contains('  ', regex=False).any(), 'espacios dobles'
    assert v['Hora'].dropna().between(0, 23).all(), 'hora fuera de rango'
    assert v['Sexo'].isin(['M', 'F', 'Sin Inf']).all(), 'sexo con valores raros'
    assert v['Mes'].nunique() == 12, 'meses mal escritos'
    return 'Validación superada'

print(validar(incidentes_limpio, victimas_limpio))
print(incidentes_limpio.equals(limpiar_incidentes(incidentes_crudo)),
      victimas_limpio.equals(limpiar_victimas(victimas_crudo)))

Validación superada
True True


Lección 3: Filtrado, ordenamiento y agrupación de datos

In [3]:
RUTA = Path("../data/processed")

incidentes = pd.read_csv(RUTA / "incidentes_limpio.csv",
                         dtype={"NRO_RADICADO": str, "NUMCOMUNA": str, "CBML": str})
victimas = pd.read_csv(RUTA / "victimas_limpio.csv",
                       dtype={"Radicado": str, "NUMCOMUNA": str})

print(incidentes.shape, victimas.shape)

(270742, 18) (235646, 20)


Filtrado con máscaras booleanas

In [4]:
es_agosto = incidentes['MES'] == 8
print(es_agosto.head(3))
agosto = incidentes[es_agosto]
print(len(agosto))

0    False
1    False
2    False
Name: MES, dtype: bool
27056


Condiciones combinadas

In [5]:
es_mes_critico = incidentes['MES'].isin([8, 9, 10])
es_periodo = incidentes['AÑO'].between(2015, 2019)
print(len(incidentes[es_mes_critico & es_periodo]))

61388


Filtrado con query

In [6]:
nocturnos_muertos = victimas.query('Hora >= 22 and Gravedad_victima == "Muertos"')
print(len(nocturnos_muertos))

168


Ordenamiento con sort_values y nlargest

In [7]:
print(victimas.nlargest(3, 'Edad')[['Edad', 'Condicion', 'Gravedad_victima']])
historial = incidentes.sort_values(['AÑO', 'MES'])
print(historial[['NRO_RADICADO', 'AÑO', 'MES', 'FECHA_ACCIDENTE']].head(3))

         Edad                   Condicion Gravedad_victima
158976  137.0                Motociclista          Heridos
211954  121.0                      Peatón          Heridos
212564  121.0  Acompañante de motocicleta          Heridos
      NRO_RADICADO   AÑO  MES      FECHA_ACCIDENTE
45762      1447957  2014    7  04/07/2014 20:00:00
45763      1448061  2014    7  04/07/2014 09:40:00
45764      1448204  2014    7  04/07/2014 19:40:00


Agrupacion con groupby

In [8]:
periodo = incidentes[incidentes['AÑO'].between(2015, 2019)]
accidentes_por_mes = periodo.groupby('MES')['NRO_RADICADO'].count()
print(accidentes_por_mes.idxmax(), accidentes_por_mes.max())
print(accidentes_por_mes.sort_values(ascending=False))

8 20846
MES
8     20846
9     20292
10    20249
5     19530
7     19492
12    19455
3     18870
11    18740
4     18419
2     18173
6     18159
1     16380
Name: NRO_RADICADO, dtype: int64


In [9]:
periodo_vic = victimas[victimas['Año'].between(2015, 2019)]

orden_meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

resumen_mes = (periodo_vic
    .groupby('Mes')
    .agg(victimas=('Radicado', 'count'),
         accidentes=('Radicado', 'nunique'),
         edad_media=('Edad', 'mean'))
    .round(1)
    .reindex(orden_meses))
resumen_mes

,victimas,accidentes,edad_media
Mes,,,
Ene,11852,9353,32.3
Feb,12647,10065,32.1
Mar,13299,10670,32.2
Abr,12677,10103,32.3
May,13141,10639,32.3
Jun,12635,10031,32.4
Jul,13825,10912,32.1
Ago,14658,11678,32.0
Sep,14295,11394,32.0


Tabla dinamica con pivot_table

In [10]:
periodo = incidentes[incidentes['AÑO'].between(2015, 2019)]

matriz = periodo.pivot_table(index='MES', columns='AÑO',
                             values='NRO_RADICADO', aggfunc='count')
print(matriz.shape)
matriz.loc[[1, 8, 9, 10]]

(12, 5)


AÑO,2015,2016,2017,2018,2019
MES,,,,,
1,3263,3322,3478,3177,3140
8,4104,4256,4220,3861,4405
9,4135,4095,4043,3721,4298
10,3976,4242,3888,3779,4364


Valores por fila con  transform

In [11]:
victimas['victimas_accidente'] = victimas.groupby(['Radicado', 'Fecha_incidente'])['Radicado'].transform('count')
graves = victimas[victimas['victimas_accidente'] >= 3]
print(len(graves), graves['Radicado'].nunique())
victimas = victimas.drop(columns='victimas_accidente')

16612 4913


Laboratorio guiado 3:perfil de habitos de compra

Paso 1: pedidos por cliente

In [12]:
victimas_por_accidente = victimas.groupby(['Radicado', 'Fecha_incidente'])['Radicado'].count()
print(len(victimas_por_accidente))
victimas_por_accidente.describe()

189114


count    189114.000000
mean          1.246026
std           0.559642
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          33.000000
Name: Radicado, dtype: float64

Paso 2: segmentar clientes con pd.cut

In [13]:
import numpy as np

segmento_accidente = pd.cut(victimas_por_accidente,
                            bins=[0, 1, 2, np.inf],
                            labels=['una víctima', 'dos víctimas', 'tres o más'])
print(segmento_accidente.value_counts().sort_index())
print((segmento_accidente.value_counts(normalize=True).sort_index() * 100).round(1))

Radicado
una víctima     149373
dos víctimas     34828
tres o más        4913
Name: count, dtype: int64
Radicado
una víctima     79.0
dos víctimas    18.4
tres o más       2.6
Name: proportion, dtype: float64


Paso 3: llevar el segmento a cada pedido con map

In [14]:
victimas = victimas.join(segmento_accidente.rename('segmento'), on=['Radicado', 'Fecha_incidente'])
victimas[['Radicado', 'Fecha_incidente', 'Condicion', 'segmento']].head()

,Radicado,Fecha_incidente,Condicion,segmento
0,1423940,1/01/2014,Motociclista,una víctima
1,1423921,1/01/2014,Motociclista,dos víctimas
2,1423921,1/01/2014,Peatón,dos víctimas
3,1423849,1/01/2014,Motociclista,dos víctimas
4,1423849,1/01/2014,Peatón,dos víctimas


Paso 4: resumen por segmento

In [15]:
resumen_segmento = (victimas
    .groupby('segmento', observed=True)
    .agg(accidentes=('Radicado', 'nunique'),
         victimas=('Radicado', 'count'),
         edad_mediana=('Edad', 'median')))
resumen_segmento

,accidentes,victimas,edad_mediana
segmento,,,
una víctima,149366,149373,29.0
dos víctimas,34827,69656,28.0
tres o más,4913,16612,28.0


Paso 5: franjas horarias y tabla cruzada

In [16]:
victimas['franja'] = pd.cut(victimas['Hora'],
                            bins=[-1, 5, 11, 17, 23],
                            labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(victimas['segmento'], victimas['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
una víctima,8.1,32.1,34.7,25.1
dos víctimas,9.8,27.4,32.5,30.3
tres o más,11.4,25.5,30.8,32.3


Paso 6: las combinaciones de día y franja más activas

In [17]:
(victimas.groupby(['Dia', 'franja'], observed=True)
         .size()
         .sort_values(ascending=False)
         .head(3))

Dia  franja
Mar  tarde     11835
Mié  tarde     11780
Sáb  tarde     11683
dtype: int64

Paso 7: dejar orders como estaba

In [18]:
victimas = victimas.drop(columns=['segmento', 'franja'])
print(victimas.shape)

(235646, 20)
